In [3]:
t = 5
t**2

25

In [2]:
import gurobipy as gp
from gurobipy import GRB

# Step 1: Create an empty model
model = gp.Model()

# Step 2: Read the LP file
model = gp.read("Error_Backward_t12_k(-103.69, 4)")  # Correct way
# Step 3: Optimize the model
model.optimize()

# Step 4: Check the result
if model.status == GRB.OPTIMAL:
    print("Optimal objective value:", model.objVal)
    
    # Optional: print variable values
    for v in model.getVars():
        print(f"{v.varName}: {v.x}")
else:
    print("Optimization was not successful. Status code:", model.status)


Set parameter Username
Set parameter LicenseID to value 2767213
Academic license - for non-commercial use only - expires 2027-01-18
Read LP format model from file Error_Backward_t12_k(-103.69, 4).lp
Reading time = 0.27 seconds
: 4108 rows, 603 columns, 356867 nonzeros
Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (win64 - Windows 11.0 (26100.2))

CPU model: 12th Gen Intel(R) Core(TM) i9-12900HK, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 4108 rows, 603 columns and 356867 nonzeros
Model fingerprint: 0x6dd16a87
Coefficient statistics:
  Matrix range     [6e-08, 4e+03]
  Objective range  [3e-01, 6e+03]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e-02, 2e+08]
Presolve removed 661 rows and 544 columns
Presolve time: 0.11s
Presolved: 3447 rows, 59 columns, 139678 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    4.5336417e+07   7.887411e+09   0.000000e+

In [ ]:
import numpy as np

objs = [14199747.33759272, 24614005.75514008, 15644322.52600235, 13840096.2747652, 15644322.52600235]
nbOS = len(objs)

mean_obj = np.mean(objs)
std_obj = np.std(objs)   # sample standard deviation
ci_low = mean_obj - 1.96 * std_obj / np.sqrt(nbOS)
ci_high = mean_obj + 1.96 * std_obj / np.sqrt(nbOS)
CI = mean_obj - ci_low

print(f"\nTwo-Stage OOS Evaluation Results")
print(f"μ ± 1.96*σ/√NS = {mean_obj:.2f} ± {CI:.2f}")
print(f"95% Confidence Interval = [{ci_low:.2f}, {ci_high:.2f}]")



In [ ]:
import numpy as np
import pandas as pd
# Show all rows (be careful with very large DataFrames!)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# Fake network processor object with needed attributes
class DummyNetworkProcessor:
    def __init__(self):
        self.V = 21  # total number of nodes (indexed 0 to 20)
        self.V3 = [14, 15, 16, 17, 18, 19, 20]  # PoD node indices
        self.T_Aft = [10, 11, 12, 13, 14, 15]   # post-landfall time periods

network_processor = DummyNetworkProcessor()

# Test state
k_t = (65.0, 3)   # Example state: cat = 4
std_pod = 1       # STD for time distribution

def msp_post_landfall_demand(k_t, network_processor):
        """
        Generate deterministic PoD-level demand for t > t_L (i.e., post-landfall),
        based on the hurricane category from the Markov state k_t at landfall time.
        """
        # Load pod info from Excel
        pod_info_df = pd.read_excel("Data/PoD location information.xlsx", sheet_name="Sheet1")
        
        # Extract hurricane category from Markov state
        cat = int(k_t[1])

        # Initialize local demand dictionary
        demand = {}

        if cat == 0 or pd.isna(cat):
            # No demand if hurricane category is 0 or NaN
            for i in network_processor.V3:
                for t in network_processor.T_Aft:
                    demand[(i, t)] = 0.0
        else:
            # Retrieve demand-related columns
            demand_col = f"I*p_c (cat {cat})"
            evac_col = "% 1- (Likely to evacuate)"
            total_demand = pod_info_df[demand_col] * pod_info_df[evac_col]
            total_demand = total_demand.to_numpy()

            # Gaussian-weighted temporal distribution
            mean = np.mean(network_processor.T_Aft)
            std = std_pod
            weights = np.array([
                np.exp(-((t - mean) ** 2) / (2 * std ** 2)) for t in network_processor.T_Aft
            ])
            weights /= weights.sum()

            # Assign demand to PoD nodes
            for idx, i in enumerate(network_processor.V3):
                for w_idx, t in enumerate(network_processor.T_Aft):
                    if (i, t) not in demand:
                        demand[(i, t)] = total_demand[idx] * weights[w_idx]

        # Assign zero demand to all non-PoD nodes
        for i in range(network_processor.V):
            if i not in network_processor.V3:
                for t in network_processor.T_Aft:
                    if (i, t) not in demand:
                        demand[(i, t)] = 0.0

        return demand

# Call the function
demand_dict = msp_post_landfall_demand(k_t, network_processor)
print(demand_dict)

df = pd.DataFrame([(i, t, d) for (i, t), d in demand_dict.items()],
                  columns=["Node", "Time", "Demand"])
df.sort_values(["Node", "Time"]).reset_index(drop=True)


In [ ]:
import json
import numpy as np
import pandas as pd
import time
np.random.seed()

def load_transition_dict(path):
    with open(path, "r") as f:
        return json.load(f)

def build_node_lists(transition_dict):
    nodeLists = {}
    for t in transition_dict:
        nodeLists[int(t)] = [eval(s) for s in transition_dict[t].keys()]
    return nodeLists

def MC_sample(t, current_state, transition_dict):
    """ Draw one next state given current time t and current_state """

    # Convert keys to strings
    t_key     = str(t)
    state_key = str(current_state)
    # Get all possible transitions
    probs_dict = transition_dict[t_key][state_key]
    # Extract valid next states and probabilities
    states_str = [s for s, p in probs_dict.items() if p > 0]
    weights    = [p for s, p in probs_dict.items() if p > 0]
    states = [eval(s) for s in states_str] # Convert state strings to tuples
    idx    = np.random.choice(len(states), p=weights) # Randomly sample next state
    return states[idx]

def create_OSpaths(nbOS, T_Bef, k_init, transition_dict, output_file):
    """
    Generate Out-of-Sample (OOS) paths using transition probabilities.
    Saves a CSV file where each cell contains a stringified tuple representing a state.
    """
    T = len(T_Bef)
    OS_paths = np.full((nbOS, T), None, dtype=object)

    # Initialize first state for all paths
    OS_paths[:, 0] = k_init

    # Generate paths using MC sampling
    for s in range(nbOS):
        for t in range(1, T):
            prev_state = OS_paths[s, t-1]
            next_state = MC_sample(t-1, prev_state, transition_dict)
            OS_paths[s, t] = next_state

    # Save the paths as CSV
    df = pd.DataFrame(OS_paths)
    df.to_csv(output_file, index=False)

    print(f"[INFO] OOS paths generated and saved to: {output_file}")
    return OS_paths

In [ ]:
import numpy as np
xval = [[0] * 5 for _ in range(3)]
xval1 = np.zeros((3, 5))
print(xval)
print(xval1)

In [ ]:
V = [0, 1, 2]
A = [(0, 1), (0, 2), (1, 2)]

for (i, j) in A:
    print("homa")

for i in range(len(V)):
    for j in range(len(V)):
        if (i, j) in A:
            print("milad")



In [ ]:
f1_prev = {}
for t in range(3):
    for i in range(2):
        for j in range(2):
            f1_prev[i, j, t] = 0.0

print(f1_prev)

for t in range(3):
    for i in range(2):
        for j in range(2):
            key = (i, j, t)
            f1_prev[(key)] = 7.0
print(f1_prev)

In [ ]:
T = 10
for t in range(9, 0, -1):
    print(t)

In [ ]:
import json

with open("Data/pi_mssp_d_ST_track_15.json", "r") as f:
    transition_dict = json.load(f)

nodeLists = {int(t): [eval(s) for s in trans.keys()] 
             for t, trans in transition_dict.items()}

# All Markov states reachable at stage t
print(nodeLists[2])

In [ ]:
import json

with open("Data/pi_mssp_d_ST_track_15.json", "r") as f:
    data = json.load(f)

print("Top-level keys (stages):", list(data.keys())[:10])
first_stage = list(data.keys())[0]
print("Sample stage:", first_stage)
print("Sample states at stage 0:", list(data[first_stage].keys())[:5])

first_state = list(data[first_stage].keys())[0]
print("Sample transitions from", first_state, ":", data[first_stage][first_state])


In [ ]:
import json

def load_transition_dict(path):
    with open(path, "r") as f:
        return json.load(f)

def build_node_lists(transition_dict):
    nodeLists = {}
    for t in transition_dict:
        nodeLists[int(t)] = [eval(s) for s in transition_dict[t].keys()]
    return nodeLists

# Load transition dictionary and build nodeLists
transition_dict = load_transition_dict("Data/pi_mssp_d_ST_track_15.json")
nodeLists = build_node_lists(transition_dict)

state_to_idx = {}
idx = 0
for t in nodeLists:
    for k in nodeLists[t]:
        state_to_idx[k] = idx
        idx += 1

K = len(state_to_idx)
print(K)

t = 8
n = list(transition_dict[str(t)].keys())[0]
print("Current state at t =", n)
print("Reachable next states at t+1:")
print(transition_dict[str(t)][n])



In [ ]:
import numpy as np

def MC_sample_json(t, current_state, transition_dict):
    """
    Return ONE next state given (t , current_state).

    Parameters
    ----------
    t : int
        Current time step (0-based, as in your forward pass).
    current_state : tuple
        Current Markov state, e.g. (23.55, 1).
    transition_dict : dict
        Loaded JSON dictionary of transition probabilities.

    Returns
    -------
    tuple
        Next state sampled according to transition probabilities.
    """
    # Keys in the JSON are strings
    t_key       = str(t)
    state_key   = str(current_state)

    probs_dict  = transition_dict[t_key][state_key]          # { "(next_state)": prob, ... }
    states_str  = [s for s, p in probs_dict.items() if p > 0]
    weights     = [p for s, p in probs_dict.items() if p > 0]

    states      = [eval(s) for s in states_str]              # back to tuple form
    idx         = np.random.choice(len(states), p=weights)   # Monte-Carlo draw
    return states[idx]


In [ ]:
# --- 2. Set known values -------------------------------------------------
t              = 1              # current time step
current_state  = (-23.55, 0)       # deterministic starting state at t = 0

# --- 3. Call the single-step sampler ------------------------------------
next_state = MC_sample_json(t, current_state, transition_dict)

print(f"Current state (t={t}): {current_state}")
print(f"Sampled next state   : {next_state}")